In [ ]:
import requests
import re
import json
from bs4 import BeautifulSoup
from wonderwords import RandomWord
import random
import datetime
from tqdm import tqdm
import pandas as pd


In [2]:


def get_video_ids(query):
    '''
    Devuelve una lista de 100 video_ids de YouTube a partir de una consulta de búsqueda
    Estos vídeos cumplen con el filtro de formato de duración media y subtítulos presentes
    '''
    url = "https://www.youtube.com/results"
    params = {"search_query": query,
              "sp": "EgQQASgB" #Filtro para videos formato media duración con subtítulos
              }
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    response = requests.get(url, params=params, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    # Buscar el script que contiene ytInitialData
    scripts = soup.find_all("script")

    for script in scripts:
        if "ytInitialData" in script.text:
            json_text = re.search(r"ytInitialData\s*=\s*(\{.*\});", script.text)
            if json_text:
                data = json.loads(json_text.group(1))
                break
    else:
        return []

    # Buscar todos los videoId dentro del JSON
    video_ids = set()
    def extract_ids(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if k == "videoId":
                    video_ids.add(v)
                else:
                    extract_ids(v)
        elif isinstance(obj, list):
            for item in obj:
                extract_ids(item)

    extract_ids(data)

    return list(video_ids)


# 🔹 Ejemplo
ids = get_video_ids("x after:2026-02-15")
print(ids)


['N7UazLGpi58', 'n-bEIjHEEc8', 'bdXDP3kQUkY', 'xHzWwttwcLE', 'JHbZuGVdtHw', 'Tfag8qgrlz4', '8HcgjqZvLuA', 'AAbyOXDwRXM', 'Tn-UIyw70mI', 'H3pvDEzmAVY', 'EGzIEh2kyWo', 'Jbi0yHThvNQ', 'qB3oE-GBuw0', 'dC-SUOL-hRc', 'mNrG4NvZA_I', 'Z3ZvxIz20-c', 'S2c3OkneiOE', '5L58OI0HadA', 'gE5mFMgvRgw', 'kKM1NvQ80fs']


In [4]:

#Esta celda de código genera una lista de "num_ids" video_ids aleatorios a partir de palabras aleatorias y la función get_video_ids.
#Por cada palabra aleatoria se realiza una búsqueda en Youtube de videos que contengan esa palabra en el título y que hayan sido publicados en el último día.
#Se planea adaptar este código a una función que reciba ciertos parámetros de búsqueda para filtrar los vídeos.

num_ids = 25

lista_palabras_aleatorias = []
lista_ids_aleatorios = []
w = RandomWord()
while len(lista_ids_aleatorios) < num_ids:
    try: 
        random_word = w.word()
        lista_ids_aleatorios.append(random.choice(get_video_ids('\"' + random_word + '\" intitle:' + random_word+" after:" + str(datetime.date.today()-datetime.timedelta(days=1)))))
        lista_palabras_aleatorias.append(random_word)
    except: pass
   

In [ ]:
def get_random_ids(num_ids=25, after_date=None, before_date=None):
    '''
    Devuelve una lista de "num_ids" video_ids aleatorios a partir de palabras aleatorias y la función get_video_ids.
    Por cada palabra aleatoria se realiza una búsqueda en Youtube de videos que contengan esa palabra en el título y que hayan sido publicados entre "after_date" y "before_date".
    '''
    ##Habría que añadir algo que controle que la fecha de inicio no sea posterior a la de fin, o que no se introduzcan fechas futuras, etc. 
    # Porque si no se mete en un bucle infinito.
    lista_palabras_aleatorias = []
    lista_ids_aleatorios = []
    w = RandomWord()
    while len(lista_ids_aleatorios) < num_ids:
        try: 
            random_word = w.word()
            query = f'\"{random_word}\" intitle:{random_word}'
            if after_date:
                query += f' after:' + str(after_date)
            if before_date:
                query += f' before:' + str(before_date)
            lista_ids_aleatorios.append(random.choice(get_video_ids(query)))
            lista_palabras_aleatorias.append(random_word)
        except: pass
    return lista_palabras_aleatorias, lista_ids_aleatorios
palabras, ids = get_random_ids(num_ids=5, after_date=datetime.datetime.now(datetime.timezone.utc)-datetime.timedelta(days=1))
print(ids)

In [ ]:
print(list(zip(lista_palabras_aleatorias,lista_ids_aleatorios)))

25
[('waffle', 'v9D9433O75A'), ('congress', 'PBQAtcgfkMU'), ('cat', 'tjgbTRFKj3c'), ('chestnut', 'DsdquWlUwUo'), ('warmth', 'pmSJbhusK0o'), ('client', 'DNPZn3ktV-w'), ('formula', 'HrlCdPTwDZY'), ('stable', 'IykcmG2rFV8'), ('agency', 'DaFYswBG810'), ('lake', 'iCEZJmEWhNU'), ('airbus', 'igYAlg_Uv3A'), ('heartbreaking', 'oJFtss8q7FQ'), ('certification', 'QjAsng141Ms'), ('roundabout', 'pnO6QJiljmo'), ('stink', 'vE4HNafvmVc'), ('surgeon', 'Pvrsn-_AaNw'), ('memorial', 'zpUjLBPVAJI'), ('rabbi', 'BsLO3PYa0FA'), ('knowledge', '19KLbLrAAAY'), ('blink', 'iUYzOxrTEt0'), ('dusty', '-1iMOuz2Zyk'), ('threshold', 'uu9eNpR2b98'), ('day', 's8eIC06afmM'), ('planet', 'TmURVjr45EY'), ('bitter', 'doWXMZKdQsc')]


In [12]:
url = f"https://www.youtube.com/watch?v={ids[2]}"
headers = {"User-Agent": "Mozilla/5.0"}

r = requests.get(url, headers=headers)

match = re.search(r"ytInitialPlayerResponse\s*=\s*(\{.*?\});", r.text)

data = json.loads(match.group(1))

data['videoDetails']

{'videoId': 'AKEciZo9Hrk',
 'title': 'Kill Bill: The Whole Bloody Affair',
 'lengthSeconds': '15187',
 'channelId': 'UClg8q3BGmVTc98KO_HIz_ig',
 'isOwnerViewing': False,
 'shortDescription': "Quentin Tarantino's KILL BILL: THE WHOLE BLOODY AFFAIR unites Volume 1 and Volume 2 into a single, unrated epic—presented exactly as he intended, complete with a new anime sequence. Uma Thurman stars as The Bride, left for dead after her former boss and lover Bill ambushes her wedding rehearsal, shooting her in the head and stealing her unborn child. To exact her vengeance, she must first hunt down the four remaining members of the Deadly Viper Assassination Squad before confronting Bill himself. With its operatic scope, relentless action, and iconic style, THE WHOLE BLOODY AFFAIR stands as one of cinema's definitive revenge sagas—rarely shown in its complete form, and now presented with a classic intermission.",
 'isCrawlable': True,
 'thumbnail': {'thumbnails': [{'url': 'https://i.ytimg.com/vi/A

In [25]:

def get_video_info(video_id,palabra_aleatoria=""):
    '''
    Devuelve un diccionario con información relevante del video de YouTube dado su video_id.
    '''
    url = f"https://www.youtube.com/watch?v={video_id}"
    headers = {"User-Agent": "Mozilla/5.0"}
    
    r = requests.get(url, headers=headers)
    if r.status_code != 200:
            return None
    
    match = re.search(r"ytInitialPlayerResponse\s*=\s*(\{.*?\});", r.text)
    
    if not match:
        return None
    
    data = json.loads(match.group(1))
    if "videoDetails" not in data.keys():
        return None
    details = data["videoDetails"]
    
    titulo = details["title"]
    canal = details["author"]
    descripcion = details["shortDescription"]
    visualizaciones = details["viewCount"] if "viewCount" in details.keys() else None
    keywords = details["keywords"] if "keywords" in details else []
    captions = "captions" in data.keys()
    short = data["microformat"]["playerMicroformatRenderer"]["isShortsEligible"]
    try:
        likes = data["microformat"]["playerMicroformatRenderer"]["likeCount"]
    except KeyError:
        likes = None
    categoria = [i.strip() for i in data["microformat"]["playerMicroformatRenderer"]["category"].split("&")]
    duracion = details["lengthSeconds"]

    
    fecha = data["microformat"]["playerMicroformatRenderer"]["publishDate"]
    
    return {
        "id": video_id,
        "palabra_aleatoria": palabra_aleatoria,
        "titulo": titulo,
        "canal": canal,
        "descripcion": descripcion,
        "visualizaciones": visualizaciones,
        "likes": likes,
        "duracion": duracion,
        "fecha_publicacion": fecha,
        "horas_desde_publicacion": (datetime.datetime.now(datetime.timezone.utc) - datetime.datetime.strptime(fecha, "%Y-%m-%dT%H:%M:%S%z")).total_seconds() / 3600,
        "categoria": categoria,
        "keywords": keywords,
        "subtitulos": captions,
        "short": short  
    }

#🔹 Ejemplo
info = get_video_info("QjAsng141Ms")
info

{'id': 'QjAsng141Ms',
 'palabra_aleatoria': '',
 'titulo': 'How to Start a Coaching Business Without Certification (For Professional Women) | Nik Scott',
 'canal': 'Nik Scott',
 'descripcion': 'How long did you climb the corporate ladder before realizing it was leaning on the wrong wall?\n\n⭐️ BUILD YOUR $2K OFFER IN 2 HOURS: https://www.herincomeedit.com/2k ⭐️ \n\n👉 [FREE] GET 101 COACHING IDEAS: https://www.herincomeedit.com/free-guide \n👉 [FREE] EMAIL COURSE: https://www.herincomeedit.com/email-course \n\nNew to my channel? Start Here 👉 https://www.youtube.com/playlist?list=PLoACzrZFQEqy38XB0-UfGfcFZ7PoEdt-F\n\n--\nYou\'ve got the MBA. The promotions. The six-figure salary. The remote position that everyone said you should be grateful for. And you\'re still waking up wondering if this is really it. Meanwhile, you\'re telling yourself you can\'t start a coaching business because you\'re not certified yet. Because you need more training. Because who are you to charge people?\n\nBut wh

In [26]:
df = pd.DataFrame()
for i in tqdm(range(len(ids))):
    df = pd.concat([df, pd.DataFrame([get_video_info(ids[i], palabras[i])])], ignore_index=True)

100%|██████████| 5/5 [00:05<00:00,  1.14s/it]


In [27]:
df

,id,palabra_aleatoria,titulo,canal,descripcion,visualizaciones,likes,duracion,fecha_publicacion,horas_desde_publicacion,categoria,keywords,subtitulos,short
0,tLhJOZgw5Ng,processor,Best Gaming Processor for BGMI? Snapdragon vs ...,Tech But Game,"Doston, aaj ki video mein hum Snapdragon aur M...",70,5,179,2026-02-15T04:31:11-08:00,28.454801,[Gaming],"[best gaming processor, bgmi, snapdragon vs me...",True,False
1,wBAQsvkxfRw,glimpse,First Glimpse of Almora 😍 Just Before Entering...,Trips n Treatss,This video was captured just before entering A...,11,1,60,2026-02-15T18:31:01-08:00,14.457971,"[Travel, Events]",[],True,False
2,AKEciZo9Hrk,bloody,Kill Bill: The Whole Bloody Affair,YouTube Movies,Quentin Tarantino's KILL BILL: THE WHOLE BLOOD...,None,0,15187,2026-02-16T03:11:12-08:00,5.788632,[Movies],[],False,False
3,s4e_aIf3CXI,devil,1st Time Reaction to The Rolling Stones - Symp...,P Reacts Daily,​ ⁨@PReactsDaily⁩ Reaction to\n\nPReactsDaily ...,30,2,990,2026-02-15T06:30:00-08:00,26.475527,[Comedy],"[#MusicReactions, #Reactions, #Singing, #Rolli...",True,False
4,zZR5D_p-W3M,elephant,Endangered Asian elephant calf born at Smithso...,DC News Now,After many fans voted for the elephant calf’s ...,96,3,38,2026-02-15T13:12:39-08:00,19.764960,"[News, Politics]",[top video],True,False


In [ ]:
df.to_csv("videos_info.csv", index=False)
